## Data Types and Null and Duplication Handling

### Create a Silver Table with Data Types

In [ ]:
%sql

-- create the silver orders table with data types typed in
CREATE OR REPLACE TABLE food_delivery_lab.silver.orders_typed AS
SELECT
  order_id,
  restaurant_id,
  driver_id,
  city,
  state,
  cuisine,
  CAST(order_value_usd AS DECIMAL(10,2)) AS order_value_usd,
  CAST(items_count AS INT) AS items_count,
  CAST(order_time AS TIMESTAMP) AS order_time,
  CAST(delivery_time AS TIMESTAMP) AS delivery_time,
  delivery_status,
  payment_method
FROM food_delivery_lab.bronze.orders;

### TIMESTAMP vs TIMESTAMP_NTZ

In [ ]:
%sql

-- see the effect of "No Timezone" v/s "Timezone"
CREATE OR REPLACE TEMP VIEW delivery_schedule AS
SELECT *
FROM VALUES
  ('D001','ORD-5001', TIMESTAMP '2026-06-10 12:00:00', TIMESTAMP_NTZ '2026-06-10 12:00:00'),
  ('D002','ORD-5005', TIMESTAMP '2026-06-10 18:30:00', TIMESTAMP_NTZ '2026-06-10 18:30:00')
AS t(schedule_id, order_id, delivery_ts, delivery_ntz);

SET TIME ZONE 'PST';

In [ ]:
%sql
SELECT * FROM delivery_schedule;

### Complex Data Types - `ARRAY`, `MAP` and `STRUCT`

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW orders_enriched AS
SELECT *
FROM VALUES
  ('ORD-5001',
    ARRAY('Extra Sauce','Napkins'),
    MAP('priority','high','channel','mobile'),
    named_struct('street','5th Ave','city','New York','zip','10001')
  ),
    
  ('ORD-5002',
    ARRAY('Cutlery'),
    MAP('priority','medium','channel','web'),
    named_struct('street','Sunset Blvd','city','Los Angeles','zip','90001')
  )
AS t(order_id, extras, metadata, address);

In [ ]:
%sql
SELECT
  order_id,
  extras[0] AS first_extra,
  metadata['priority'] AS priority,
  address.city AS city
FROM orders_enriched;

### Handle Duplicate and Missing Values

In [ ]:
%sql

-- find duplicate orders
SELECT order_id, COUNT(*)
FROM food_delivery_lab.bronze.orders
GROUP BY order_id
HAVING COUNT(*) > 1;

### PySpark Deduplication

In [ ]:
from pyspark.sql.functions import col
df = spark.table("food_delivery_lab.bronze.orders")


print("Before:", df.count())

dedup_df = df.orderBy(col("order_time").desc()) \
             .dropDuplicates(["order_id"])

print("After:", dedup_df.count())

display(dedup_df)

### Handle Nulls

In [ ]:
clean_df = df.dropna(subset=["order_value_usd"]) \
    .fillna({
        "items_count": 1,
        "delivery_status": "Unknown"
    })

print("Before:", df.count())
print("After:", clean_df.count())

display(clean_df)

### Overwrite Silver Table after handling Nulls

In [ ]:
from pyspark.sql.functions import col

clean_typed_df = clean_df.select(
    col("order_id"),
    col("restaurant_id"),
    col("driver_id"),
    col("city"),
    col("state"),
    col("cuisine"),
    col("order_value_usd").cast("decimal(10,2)"),
    col("items_count").cast("int"),
    col("order_time").cast("timestamp"),
    col("delivery_time").cast("timestamp"),
    col("delivery_status"),
    col("payment_method")
)

clean_typed_df.write \
    .mode("overwrite") \
    .saveAsTable("food_delivery_lab.silver.orders_typed")


In [ ]:
%sql

-- you should now see rows without null values in the order_value_usd column
SELECT * FROM food_delivery_lab.silver.orders_typed;